In [ ]:
!pip install groq -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""

In [ ]:
import os
import json
import numpy as np
import joblib
from scipy.spatial.distance import cdist
from groq import Groq

PROJECT_DIR = '/content/drive/MyDrive/fraud_detection_project'
EPS = 0.30
FEATURE_COLS = ['avg_amount', 'std_amount', 'max_avg_ratio', 'txn_count',
                'txn_per_active_day', 'dispersion_index']

FEATURE_PROMPTS = {
    'avg_amount':          "Average incoming transaction amount",
    'std_amount':          "Std deviation of incoming transaction amounts",
    'max_avg_ratio':       "Ratio of max transaction to average transaction",
    'txn_count':           "Total number of incoming transactions",
    'txn_per_active_day':  "Transactions per active day",
    'dispersion_index':    "Dispersion index of transaction timing",
}

scaler = joblib.load(f'{PROJECT_DIR}/fitted_scaler.pkl')
pca    = joblib.load(f'{PROJECT_DIR}/fitted_pca.pkl')
core_points = np.load(f'{PROJECT_DIR}/core_points.npy')
clf    = joblib.load(f'{PROJECT_DIR}/fraud_classifier_xgb.pkl')

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"

SYSTEM_PROMPT = """You are a fraud-review assistant for a payments platform.
You receive TWO independent signals about a recipient account, plus its raw
behavioral features:
1. An unsupervised anomaly signal (distance from known normal-pattern accounts,
   from DBSCAN clustering)
2. A supervised fraud probability (from a trained classifier)

You do NOT have access to raw transaction data or personal information.
You NEVER approve, block, or move money -- only produce a structured risk
assessment for a human reviewer. If the two signals disagree, say so explicitly
and explain what that disagreement might mean.

Respond with ONLY valid JSON, no markdown fences, no preamble:
{
  "risk_level": "low" | "medium" | "high",
  "explanation": "1-3 sentences referencing both signals and key feature values",
  "signals_agree": true | false,
  "recommended_action": "monitor" | "alert" | "escalate" | "hold_for_review",
  "confidence": "low" | "medium" | "high"
}
"""


def get_dbscan_signal(raw_features: dict) -> dict:
    x_raw = np.array([[raw_features[c] for c in FEATURE_COLS]])
    for col in ['txn_count', 'txn_per_active_day', 'dispersion_index']:
        idx = FEATURE_COLS.index(col)
        x_raw[0, idx] = np.log1p(x_raw[0, idx])
    x_scaled = scaler.transform(x_raw)
    x_pca = pca.transform(x_scaled)
    min_dist = float(cdist(x_pca, core_points).min())
    return {"is_anomalous": min_dist > EPS, "distance": min_dist}


def get_classifier_signal(raw_features: dict) -> dict:
    x = np.array([[raw_features[c] for c in FEATURE_COLS]])
    x_log = x.copy()
    for col in ['txn_count', 'txn_per_active_day', 'dispersion_index']:
        idx = FEATURE_COLS.index(col)
        x_log[0, idx] = np.log1p(x_log[0, idx])
    prob = float(clf.predict_proba(x_log)[0, 1])
    return {"fraud_probability": prob}


def get_agent_verdict(account_id: str, raw_features: dict,
                       dbscan_signal: dict, clf_signal: dict) -> dict:
    feat_lines = "\n".join(f"- {k}: {v:.4f}" for k, v in raw_features.items())
    user_prompt = f"""Account ID: {account_id}

Features:
{feat_lines}

Signal 1 (DBSCAN, unsupervised): is_anomalous={dbscan_signal['is_anomalous']}, distance_to_nearest_core={dbscan_signal['distance']:.3f} (threshold={EPS})
Signal 2 (XGBoost classifier, supervised): fraud_probability={clf_signal['fraud_probability']:.4f}

Assess this account's fraud risk using both signals."""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_prompt}],
        temperature=0.2,
        max_tokens=1000,
    )
    raw = resp.choices[0].message.content.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    result = json.loads(raw)
    result["account_id"] = account_id
    return result


def score_account(account_id: str, raw_features: dict) -> dict:
    """Full pipeline: DBSCAN + classifier + agent, one call."""
    dbscan_signal = get_dbscan_signal(raw_features)
    clf_signal = get_classifier_signal(raw_features)
    verdict = get_agent_verdict(account_id, raw_features, dbscan_signal, clf_signal)
    verdict["dbscan_distance"] = dbscan_signal["distance"]
    verdict["dbscan_is_anomalous"] = dbscan_signal["is_anomalous"]
    verdict["classifier_probability"] = clf_signal["fraud_probability"]
    return verdict




def engineer_features(amounts: list[float], timestamps_hours: list[float]) -> dict:
    """Reproduces Stage 1-2's per-account feature engineering on raw txns.
    amounts: list of transaction amounts for this account.
    timestamps_hours: list of transaction times in hours since account's
                       first transaction (must be same length as amounts).
    """
    amounts = np.array(amounts, dtype=float)
    times   = np.sort(np.array(timestamps_hours, dtype=float))

    txn_count  = len(amounts)
    avg_amount = amounts.mean()
    std_amount = amounts.std(ddof=0) if txn_count > 1 else 0.0
    max_avg_ratio = amounts.max() / avg_amount if avg_amount > 0 else 0.0

    active_days = max((times.max() - times.min()) / 24.0, 1.0)
    txn_per_active_day = txn_count / active_days

    if txn_count > 2:
        gaps = np.diff(times)
        dispersion_index = gaps.var(ddof=0) / gaps.mean() if gaps.mean() > 0 else 0.0
    else:
        dispersion_index = 0.0

    return {
        'avg_amount': float(avg_amount),
        'std_amount': float(std_amount),
        'max_avg_ratio': float(max_avg_ratio),
        'txn_count': float(txn_count),
        'txn_per_active_day': float(txn_per_active_day),
        'dispersion_index': float(dispersion_index),
    }




def print_verdict(account_id: str, raw_features: dict, verdict: dict) -> None:
    dist        = verdict["dbscan_distance"]
    is_anom     = verdict["dbscan_is_anomalous"]
    prob        = verdict["classifier_probability"]
    risk        = verdict["risk_level"].upper()
    agree       = verdict["signals_agree"]
    action      = verdict["recommended_action"]
    conf        = verdict["confidence"]
    explanation = verdict["explanation"]

    risk_tag = {"LOW": "LOW", "MEDIUM": "MEDIUM", "HIGH": "HIGH"}.get(risk, risk)

    print("\n" + "=" * 60)
    print(f"FRAUD RISK ASSESSMENT — Account: {account_id}")
    print("=" * 60)

    print("\nInput features:")
    for col in FEATURE_COLS:
        print(f"  {FEATURE_PROMPTS[col]:<45} {raw_features[col]}")

    print("\nSignal 1 — DBSCAN (unsupervised anomaly detection)")
    print(f"  distance to nearest normal cluster : {dist:.3f}  (flag threshold = {EPS})")
    print(f"  flagged as anomalous                : {'YES' if is_anom else 'no'}")

    print("\nSignal 2 — XGBoost classifier (supervised)")
    print(f"  predicted fraud probability         : {prob:.2%}")

    print("\nDo the two signals agree?")
    print(f"  {'YES — both point the same direction' if agree else 'NO — signals conflict, needs closer look'}")

    print("\n" + "-" * 60)
    print(f"FINAL DECISION")
    print("-" * 60)
    print(f"  Risk level          : {risk_tag}")
    print(f"  Recommended action  : {action.replace('_', ' ').upper()}")
    print(f"  Confidence          : {conf.upper()}")
    print(f"  Reasoning           : {explanation}")
    print("=" * 60 + "\n")


if __name__ == "__main__":
    account_id = "C_TEST_FRAUD_001"

    amounts = [500, 45000, 91000, 88000]
    timestamps = [0, 2, 26, 27.5]

    raw_features = engineer_features(amounts, timestamps)

    print("--- Derived features (Stage 1-2 output) ---")
    for col in FEATURE_COLS:
        print(f"  {FEATURE_PROMPTS[col]:<45} {raw_features[col]:.4f}")

    verdict = score_account(account_id, raw_features)
    print_verdict(account_id, raw_features, verdict)

--- Derived features (Stage 1-2 output) ---
  Average incoming transaction amount           56125.0000
  Std deviation of incoming transaction amounts 36912.6926
  Ratio of max transaction to average transaction 1.6214
  Total number of incoming transactions         4.0000
  Transactions per active day                   3.4909
  Dispersion index of transaction timing        12.0061


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



FRAUD RISK ASSESSMENT — Account: C_TEST_FRAUD_001

Input features:
  Average incoming transaction amount           56125.0
  Std deviation of incoming transaction amounts 36912.69260024253
  Ratio of max transaction to average transaction 1.621380846325167
  Total number of incoming transactions         4.0
  Transactions per active day                   3.490909090909091
  Dispersion index of transaction timing        12.006060606060608

Signal 1 — DBSCAN (unsupervised anomaly detection)
  distance to nearest normal cluster : 2.324  (flag threshold = 0.3)
  flagged as anomalous                : YES

Signal 2 — XGBoost classifier (supervised)
  predicted fraud probability         : 0.16%

Do the two signals agree?
  NO — signals conflict, needs closer look

------------------------------------------------------------
FINAL DECISION
------------------------------------------------------------
  Risk level          : MEDIUM
  Recommended action  : HOLD FOR REVIEW
  Confidence          :